# AI Agent Memory & Context Engineering Platform

A practical notebook demonstrating structured memory capture, retrieval, ranking, context budgeting, updates, export, and deletion. Uses synthetic examples and a local in-memory store; no external LLM or user account is connected.

## 1. Setup

In [ ]:
# If needed: %pip install numpy pandas matplotlib
from dataclasses import dataclass, asdict
from datetime import datetime, timezone, timedelta
import json, re, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt


## 2. Define memory schema and store

In [ ]:
@dataclass
class Memory:
    memory_id: str
    text: str
    category: str
    created_at: str
    importance: float
    confidence: float
    source: str = "synthetic_conversation"
    status: str = "active"
    supersedes: str = ""

class MemoryStore:
    def __init__(self): self.records={}
    def add(self,m): self.records[m.memory_id]=m
    def active(self): return [m for m in self.records.values() if m.status=="active"]
    def update(self,mid,**changes):
        for k,v in changes.items(): setattr(self.records[mid],k,v)
    def delete(self,mid): return self.records.pop(mid,None) is not None
    def frame(self): return pd.DataFrame([asdict(m) for m in self.records.values()])

store=MemoryStore()


## 3. Capture example memories

In [ ]:
events=[
("m01","I prefer Python examples when learning AI engineering.","preference",.85,.98,4),
("m02","My current project is an AI meeting assistant built with Python.","project",.90,.95,2),
("m03","For code demos, I prefer small reproducible notebooks.","preference",.80,.96,10),
("m04","I am exploring agentic workflows and tool calling.","interest",.70,.90,20),
("m05","The meeting assistant should export action items as JSON.","requirement",.88,.94,1),
("m06","Email follow-up drafts must require human approval before sending.","constraint",.99,.99,1)]
now=datetime.now(timezone.utc)
for mid,txt,cat,imp,conf,age in events:
    store.add(Memory(mid,txt,cat,(now-timedelta(days=age)).isoformat(),imp,conf))
display(store.frame())


## 4. Retrieve and rank relevant memories

In [ ]:
STOP={"the","a","an","and","or","to","for","of","with","is","are","i","my","it","in","on"}
def tokens(s): return {w for w in re.findall(r"[a-z0-9]+",s.lower()) if w not in STOP}
def similarity(q,t):
    a,b=tokens(q),tokens(t)
    return len(a&b)/math.sqrt(len(a)*len(b)) if a and b else 0
def retrieve(query,top_k=6,half_life=30):
    now=datetime.now(timezone.utc); out=[]
    for m in store.active():
        age=max(0,(now-datetime.fromisoformat(m.created_at)).total_seconds()/86400)
        rel=similarity(query,m.text); rec=0.5**(age/half_life)
        score=.60*rel+.25*m.importance+.15*rec
        out.append({"memory":m,"relevance":rel,"recency":rec,"score":score})
    return sorted(out,key=lambda x:x["score"],reverse=True)[:top_k]
query="How should the meeting assistant handle action items and email follow-up?"
hits=retrieve(query)
display(pd.DataFrame([{"id":h["memory"].memory_id,"text":h["memory"].text,"score":round(h["score"],3)} for h in hits]))


## 5. Assemble context within a token budget

In [ ]:
def estimate_tokens(text):
    # Rough word-based estimate; replace with the target model tokenizer in production.
    return max(1,math.ceil(len(text.split())*1.3))
def build_context(query,budget=70):
    selected=[]; used=0
    for h in retrieve(query,top_k=20):
        m=h["memory"]; line=f"[{m.category}] {m.text}"; cost=estimate_tokens(line)
        if used+cost<=budget:
            selected.append({"id":m.memory_id,"content":line,"estimated_tokens":cost})
            used+=cost
    return {"query":query,"budget":budget,"estimated_tokens_used":used,"memories":selected}
context=build_context(query)
print(json.dumps(context,indent=2))


## 6. Update and supersede a memory

In [ ]:
# Add a newer constraint and mark the older memory as superseded.
store.update("m06",status="superseded")
replacement=Memory("m07","Email drafts can be generated, but sending always needs explicit human approval.",
                   "constraint",datetime.now(timezone.utc).isoformat(),1.0,.99,
                   supersedes="m06")
store.add(replacement)
display(store.frame()[["memory_id","text","status","supersedes"]])


## 7. Validate memory quality

In [ ]:
def validate(m):
    problems=[]
    if not m.text.strip(): problems.append("empty_text")
    if not 0<=m.importance<=1: problems.append("importance_out_of_range")
    if not 0<=m.confidence<=1: problems.append("confidence_out_of_range")
    if not m.category: problems.append("missing_category")
    return problems
display(pd.DataFrame([{"id":m.memory_id,"issues":validate(m)} for m in store.records.values()]))


## 8. Visualize memory categories and ranking

In [ ]:
active=pd.DataFrame([asdict(m) for m in store.active()])
active.category.value_counts().plot(kind="bar",figsize=(7,3),title="Active memories by category")
plt.ylabel("Count"); plt.tight_layout(); plt.show()
ranked=retrieve("Python notebook AI agent project",top_k=10)
plt.figure(figsize=(7,3)); plt.bar([x["memory"].memory_id for x in ranked],[x["score"] for x in ranked])
plt.title("Memory retrieval scores"); plt.ylabel("Composite score"); plt.tight_layout(); plt.show()


## 9. Export and delete

In [ ]:
def export_active(path="memory_export.json"):
    with open(path,"w",encoding="utf-8") as f:
        json.dump([asdict(m) for m in store.active()],f,indent=2)
    return path
print("Exported:",export_active())
print("Deleted m04:",store.delete("m04"))
print("Records remaining:",len(store.records))


## 10. Production architecture and limitations

**Production enhancements:** persistent encrypted storage; tenant isolation; embeddings and vector search; hybrid retrieval and reranking; model-specific tokenization; memory consolidation and contradiction handling; audit logs; retention, export, correction, and deletion controls; retrieval-quality evaluation.

**Limitations:** this demo uses synthetic data, lexical overlap rather than semantic embeddings, approximate token counting, and an in-memory store. It does not connect to LangChain, LangGraph, a vector database, or a real LLM. Data is not persistent across notebook sessions. Avoid storing sensitive information without a clear need and appropriate authorization.